# Lesson 16: Semi-Supervised Learning

## Introduction

This course has treated "labeled" and "unlabeled" as a strict binary — clustering, dimensionality reduction, and every other method here assumed *zero* labels. Real projects usually sit in between: a handful of expensively-labeled examples alongside a huge pool of unlabeled data that costs nothing to collect. **Semi-supervised learning** is the bridge — using the unlabeled data's structure to make the few labels go further.

This is the final lesson of the curriculum, and it closes on an honest note: semi-supervised methods are not unconditionally better than plain supervised learning on the labeled subset alone. They can help enormously, they can barely move the needle, and — through a specific, well-understood failure mode — they can actively hurt. This lesson measures all three outcomes rather than picking only the flattering one.

In this lesson:
1. Frame why semi-supervised learning exists — labeling cost vs abundant unlabeled data
2. Apply label propagation/spreading — a graph-based method that spreads labels through unlabeled data via similarity
3. Apply self-training — iteratively retrain on your own most-confident predictions
4. Sweep the labeled-data fraction to see where each method wins, ties, or loses against a supervised-only baseline
5. Derive co-training from scratch, including the class-imbalance collapse failure it's specifically designed to avoid
6. Synthesize a decision guide for when semi-supervised learning helps

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Why Semi-Supervised Learning](#framing)
4. [Label Propagation and Self-Training vs Supervised-Only](#comparison)
5. [When Self-Training Actively Hurts](#self-training-failure)
6. [Co-Training From Scratch](#co-training)
7. [Decision Guide](#decision-guide)
8. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="framing"></a>
## Why Semi-Supervised Learning

Labeling is expensive — a radiologist annotating scans, a linguist tagging parse trees, a domain expert categorizing support tickets. Collecting *unlabeled* examples is usually nearly free by comparison (more scans, more raw text, more tickets). Semi-supervised learning asks: can the unlabeled data's structure — where the natural clusters and manifolds are — help a classifier trained on a small labeled set generalize better than the labeled set alone would allow?

Two families, tested below:
- **Label propagation / spreading**: build a similarity graph over *all* data (labeled and unlabeled), then let labels "spread" from labeled nodes to their unlabeled neighbors.
- **Self-training**: train a classifier on the labeled data, use it to predict the unlabeled data, add its most confident predictions to the training set as if they were real labels, repeat.

<a name="comparison"></a>
## Label Propagation and Self-Training vs Supervised-Only

The classic sklearn benchmark dataset for this comparison: handwritten digits (8×8 images, 10 classes) — genuinely more challenging than a synthetic 2D dataset, and small enough that even 2% labeled means only ~2-3 examples per class. We sweep the labeled fraction and compare three approaches at each level.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

labeled_fractions = [0.02, 0.05, 0.1, 0.2, 0.4]
results = {'Supervised-only': [], 'Label spreading': [], 'Self-training': []}

for fraction in labeled_fractions:
    n_labeled = max(20, int(len(X_train) * fraction))
    # Stratified so every class gets at least ~2 labeled examples even at 2% -- plain random
    # sampling can otherwise miss a class entirely at the lowest fraction, breaking both
    # LabelSpreading's per-class normalization and SelfTrainingClassifier's internal CV.
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=n_labeled, random_state=42)
    labeled_idx, _ = next(splitter.split(X_train, y_train))

    y_train_semi = np.full(len(X_train), -1)
    y_train_semi[labeled_idx] = y_train[labeled_idx]

    clf_supervised = SVC(kernel='rbf', gamma='scale')
    clf_supervised.fit(X_train[labeled_idx], y_train[labeled_idx])
    results['Supervised-only'].append(accuracy_score(y_test, clf_supervised.predict(X_test)))

    label_spreading = LabelSpreading(kernel='knn', n_neighbors=10, max_iter=3000)
    label_spreading.fit(X_train, y_train_semi)
    results['Label spreading'].append(accuracy_score(y_test, label_spreading.predict(X_test)))

    # cv capped at 2: the lowest labeled fraction leaves only ~2 examples per class, too few for
    # CalibratedClassifierCV's default 5-fold internal CV
    base_estimator = CalibratedClassifierCV(SVC(kernel='rbf', gamma='scale'), ensemble=False, cv=2)
    self_training = SelfTrainingClassifier(base_estimator, threshold=0.8)
    self_training.fit(X_train, y_train_semi)
    results['Self-training'].append(accuracy_score(y_test, self_training.predict(X_test)))

results_df = pd.DataFrame(results, index=[f'{f:.0%}' for f in labeled_fractions])
results_df.index.name = 'Labeled fraction'
print(results_df.round(3))

fig, ax = plt.subplots(figsize=(9, 6))
for method in results:
    ax.plot(labeled_fractions, results[method], marker='o', label=method)
ax.set_xlabel('Fraction of training data labeled')
ax.set_ylabel('Test accuracy')
ax.set_title('Semi-Supervised Methods vs Supervised-Only (Digits)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Label spreading wins decisively at 2-10% labeled — exploiting the full unlabeled manifold structure when labels are scarcest. Self-training's gains are much smaller and inconsistent. By 40% labeled, supervised-only alone is already strong enough that the semi-supervised methods stop adding value, and label spreading even slips slightly behind")

<a name="self-training-failure"></a>
## When Self-Training Actively Hurts

Self-training's core risk: if the initial classifier's confident predictions are wrong, the model **reinforces its own mistakes** with each iteration. This is most visible when classes genuinely overlap and the initial labeled sample is small and unlucky.

In [ ]:
X_overlap, y_overlap = make_blobs(n_samples=1000, centers=2, cluster_std=5.5, random_state=7)
X_train_overlap, X_test_overlap, y_train_overlap, y_test_overlap = train_test_split(
    X_overlap, y_overlap, test_size=0.3, random_state=42, stratify=y_overlap
)

n_labeled = 8
rng = np.random.RandomState(6)
labeled_idx = rng.choice(len(X_train_overlap), size=n_labeled, replace=False)
y_train_semi_overlap = np.full(len(X_train_overlap), -1)
y_train_semi_overlap[labeled_idx] = y_train_overlap[labeled_idx]

clf_supervised = LogisticRegression().fit(X_train_overlap[labeled_idx], y_train_overlap[labeled_idx])
acc_supervised = accuracy_score(y_test_overlap, clf_supervised.predict(X_test_overlap))

self_training = SelfTrainingClassifier(LogisticRegression(), threshold=0.6, max_iter=20)
self_training.fit(X_train_overlap, y_train_semi_overlap)
acc_self_training = accuracy_score(y_test_overlap, self_training.predict(X_test_overlap))

print(f"Heavily overlapping classes, only {n_labeled} initial labels:")
print(f"  Supervised-only accuracy: {acc_supervised:.3f}")
print(f"  Self-training accuracy:   {acc_self_training:.3f}")
print(f"\n💡 With so few, possibly-unrepresentative initial labels and genuinely overlapping classes, self-training's confident pseudo-labels are wrong often enough that reinforcing them makes the final classifier WORSE than just trusting the tiny labeled set alone")

<a name="co-training"></a>
## Co-Training From Scratch

**Co-training** (Blum & Mitchell, 1998) uses **two independent views** of the same data — two feature subsets that are each individually somewhat informative. Two classifiers, one per view, each label the unlabeled pool; each classifier's most confident predictions get added to *both* views' training sets, on the assumption that each view can teach the other something it doesn't already know.

No production library implements co-training directly (it's less standardized than label propagation or self-training) — this is a genuine from-scratch implementation, not a wrapper around an existing method.

In [ ]:
def make_two_view_data(mean_gap: float, noise_std: float, n: int = 1000, seed: int = 42):
    rng = np.random.RandomState(seed)
    y = rng.randint(0, 2, n)
    view1 = np.column_stack([
        np.where(y == 0, rng.normal(-mean_gap, noise_std, n), rng.normal(mean_gap, noise_std, n)),
        rng.normal(0, 1, n),
    ])
    view2 = np.column_stack([
        np.where(y == 0, rng.normal(-mean_gap, noise_std, n), rng.normal(mean_gap, noise_std, n)),
        rng.normal(0, 1, n),
    ])
    return view1, view2, y


def co_training(view1, view2, labeled_idx, unlabeled_idx, y_seed, k_per_class: int, balanced: bool, max_rounds: int = 200):
    """Co-training with optional per-class-balanced pseudo-label selection."""
    pseudo_labels = {i: y_seed[i] for i in labeled_idx}
    remaining = list(unlabeled_idx)

    for _ in range(max_rounds):
        if not remaining:
            break
        cur_idx = np.array(list(pseudo_labels.keys()))
        cur_y = np.array([pseudo_labels[i] for i in cur_idx])
        if len(np.unique(cur_y)) < 2:
            break

        clf1 = LogisticRegression().fit(view1[cur_idx], cur_y)
        clf2 = LogisticRegression().fit(view2[cur_idx], cur_y)

        newly_labeled = set()
        for clf, view in [(clf1, view1), (clf2, view2)]:
            proba = clf.predict_proba(view[remaining])
            pred = clf.predict(view[remaining])

            if balanced:
                for cls in [0, 1]:
                    cls_positions = np.where(pred == cls)[0]
                    if len(cls_positions) == 0:
                        continue
                    top = cls_positions[np.argsort(-proba[cls_positions, cls])[:k_per_class]]
                    for i in top:
                        idx = remaining[i]
                        if idx not in pseudo_labels:
                            pseudo_labels[idx] = cls
                            newly_labeled.add(idx)
            else:
                top = np.argsort(-proba.max(axis=1))[:2 * k_per_class]
                for i in top:
                    idx = remaining[i]
                    if idx not in pseudo_labels:
                        pseudo_labels[idx] = pred[i]
                        newly_labeled.add(idx)

        if not newly_labeled:
            break
        remaining = [i for i in remaining if i not in newly_labeled]

    return pseudo_labels


view1, view2, y = make_two_view_data(mean_gap=1.3, noise_std=1.2)
X_combined = np.column_stack([view1, view2])
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42, stratify=y)

n_labeled = 20
rng = np.random.RandomState(1)
labeled_idx = rng.choice(idx_train, size=n_labeled, replace=False)
unlabeled_idx = np.array([i for i in idx_train if i not in set(labeled_idx)])

clf_supervised = LogisticRegression().fit(X_combined[labeled_idx], y[labeled_idx])
acc_supervised = accuracy_score(y[idx_test], clf_supervised.predict(X_combined[idx_test]))
print(f"Supervised-only (combined views, {n_labeled} labels): {acc_supervised:.3f}")

### The Naive Version Collapses

Adding whichever points *either* classifier is most confident about, with no regard for predicted class, can spiral: once a few wrong pseudo-labels creep in near the decision boundary, both classifiers start agreeing on the same (wrong) side more and more, and the pseudo-labeled set collapses toward a single class.

In [ ]:
naive_pseudo_labels = co_training(view1, view2, labeled_idx, unlabeled_idx, y, k_per_class=5, balanced=False)

naive_idx = np.array(list(naive_pseudo_labels.keys()))
naive_y = np.array([naive_pseudo_labels[i] for i in naive_idx])
naive_clf = LogisticRegression().fit(X_combined[naive_idx], naive_y)
naive_acc = accuracy_score(y[idx_test], naive_clf.predict(X_combined[idx_test]))

pseudo_only_naive = [i for i in naive_pseudo_labels if i not in labeled_idx]
naive_pseudo_acc = accuracy_score(y[pseudo_only_naive], [naive_pseudo_labels[i] for i in pseudo_only_naive])

print(f"Naive co-training (no class balancing):")
print(f"  Final class balance of pseudo-labeled set: {np.bincount(naive_y)}")
print(f"  Pseudo-label accuracy: {naive_pseudo_acc:.3f}")
print(f"  Final test accuracy: {naive_acc:.3f} (supervised-only was {acc_supervised:.3f})")
print("\n💡 The pseudo-labeled set has collapsed toward one class — confirmation bias compounding round over round, exactly the failure this lesson's self-training example showed in miniature")

### The Balanced Version Recovers

In [ ]:
balanced_pseudo_labels = co_training(view1, view2, labeled_idx, unlabeled_idx, y, k_per_class=3, balanced=True)

balanced_idx = np.array(list(balanced_pseudo_labels.keys()))
balanced_y = np.array([balanced_pseudo_labels[i] for i in balanced_idx])
balanced_clf = LogisticRegression().fit(X_combined[balanced_idx], balanced_y)
balanced_acc = accuracy_score(y[idx_test], balanced_clf.predict(X_combined[idx_test]))

pseudo_only_balanced = [i for i in balanced_pseudo_labels if i not in labeled_idx]
balanced_pseudo_acc = accuracy_score(y[pseudo_only_balanced], [balanced_pseudo_labels[i] for i in pseudo_only_balanced])

print(f"Balanced co-training (equal top-k per predicted class, per view):")
print(f"  Final class balance of pseudo-labeled set: {np.bincount(balanced_y)}")
print(f"  Pseudo-label accuracy: {balanced_pseudo_acc:.3f}")
print(f"  Final test accuracy: {balanced_acc:.3f} (supervised-only was {acc_supervised:.3f})")
print("\n💡 Requiring an equal number of newly-confident examples per predicted class — the balancing rule from Blum & Mitchell's original algorithm — prevents the collapse and delivers a genuine, if modest, improvement over the supervised-only baseline")

<a name="decision-guide"></a>
## Decision Guide

| | Label Propagation/Spreading | Self-Training | Co-Training |
|---|---|---|---|
| **Mechanism** | Spread labels via a similarity graph over all data | Iteratively trust your own confident predictions | Two views' classifiers teach each other |
| **Needs** | A meaningful similarity/distance metric across the whole dataset | Any classifier with confidence scores | Two conditionally-independent, individually-informative feature views |
| **This lesson's evidence** | Won decisively at 2-10% labeled on digits (up to +40 points over supervised-only) | Modest, inconsistent gains; can lose to supervised-only | Naive version collapses; balanced version gives a genuine, modest gain |
| **Biggest risk** | Similarity graph doesn't reflect true class structure | Confirmation bias — reinforcing early wrong predictions | Same confirmation-bias risk, amplified without class balancing |
| **Best used when** | Very few labels, but the unlabeled manifold has real structure to exploit | Labels are already reasonably plentiful and classes are well-separated | Genuinely independent feature views exist (e.g. text + metadata, audio + video) |

**Rule of thumb**: semi-supervised methods help most when labels are scarcest and the unlabeled data's structure is trustworthy — exactly where this lesson's digits sweep showed the largest gains. As labeled data grows, the supervised-only baseline catches up and eventually the extra machinery adds risk without adding value.

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Semi-supervised learning bridges the gap between abundant unlabeled data and scarce labels** — exploiting one to make the other go further.
2. **Label spreading exploits the full dataset's manifold structure** and showed the largest, most consistent gains here, especially when labels were extremely scarce (2-10%).
3. **Self-training's gains are real but inconsistent** — and confirmation bias is a genuine risk, demonstrated directly: with few, unlucky initial labels and overlapping classes, self-training scored *below* the supervised-only baseline.
4. **Co-training needs its class-balancing safeguard to work at all** — the naive version collapsed toward a single class through compounding confirmation bias; the balanced version (matching Blum & Mitchell's original design) recovered a genuine, if modest, improvement.
5. **Every semi-supervised gain shrinks as labeled data grows** — these methods are tools for the scarce-label regime, not a free, unconditional accuracy boost.

### When to Use Semi-Supervised Learning

- ✅ Labels are expensive and scarce, unlabeled data is abundant and its structure is trustworthy
- ✅ Two genuinely independent, informative feature views exist (co-training specifically)
- ❌ Classes are heavily overlapping with very few initial labels — self-training's confirmation-bias risk is highest exactly here
- ❌ Enough labeled data already exists that a supervised-only baseline is already strong — the added complexity isn't earning its keep

### Closing the Curriculum

This closes Lesson 16 and the full unsupervised learning curriculum — from foundational clustering (Lesson 0) through the professional-practice synthesis lessons (13-16). Every method has been derived from first principles, implemented from scratch, cross-checked against production libraries, and evaluated with evidence rather than assumption — including, in this final lesson, evidence of where the methods themselves fall short.

Sixteen lessons, thirty notebooks, one consistent standard: derive it, implement it, verify it, and be honest about where it doesn't work 🎯